# Accepted Loan Baseline Modeling

Train and evaluate baseline classifiers for accepted-loan `target_bad` prediction using the model-ready outputs from `Modeling/Preprocessing`.

Models implemented:

- Logistic Regression
- HistGradientBoostingClassifier
- Random Forest

The notebook uses the chronological train/validation/test split created during preprocessing.

## 1. Configuration

Define paths, modeling controls, and output folders. Random Forest uses a stratified training sample by default to keep runtime practical; validation and test evaluation always use full chronological splits.

In [1]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/lendingclub_mplconfig")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 200)

RANDOM_STATE = 42
RF_TRAIN_SAMPLE_ROWS = 300_000
LOGISTIC_MAX_ITER = 300
HGB_MAX_ITER = 150
RF_N_ESTIMATORS = 120
RF_MAX_DEPTH = 14

DEFAULT_PROJECT_ROOT = Path(
    "/Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/"
    "Final_Project/Final/CreditRiskRAG"
)

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "Modeling").exists() and (candidate / "README.md").exists():
            return candidate
    return DEFAULT_PROJECT_ROOT

PROJECT_ROOT = find_project_root()
PREPROCESSING_DATASET_DIR = PROJECT_ROOT / "Modeling" / "Preprocessing" / "preprocessing_outputs" / "datasets"
MODELING_OUTPUT_ROOT = PROJECT_ROOT / "Modeling" / "modeling_outputs"
TABLE_DIR = MODELING_OUTPUT_ROOT / "tables"
PLOT_DIR = MODELING_OUTPUT_ROOT / "plots"
MODEL_DIR = MODELING_OUTPUT_ROOT / "models"
for path in [TABLE_DIR, PLOT_DIR, MODEL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Preprocessing datasets:", PREPROCESSING_DATASET_DIR)
print("Modeling outputs:", MODELING_OUTPUT_ROOT)

PROJECT_ROOT: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG
Preprocessing datasets: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/Preprocessing/preprocessing_outputs/datasets
Modeling outputs: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs


## 2. Load Baseline Preprocessed Data

Use the baseline feature set first. This excludes `mths_since_last_record` and includes missingness indicators for selected informative-missing fields.

In [2]:
def save_table(df: pd.DataFrame, name: str, index: bool = False) -> Path:
    path = TABLE_DIR / f"{name}.csv"
    df.to_csv(path, index=index)
    print("Saved:", path)
    return path

def load_parquet(name: str) -> pd.DataFrame:
    path = PREPROCESSING_DATASET_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Missing preprocessing dataset: {path}")
    return pd.read_parquet(path)

X_train = load_parquet("baseline_train_X.parquet")
X_valid = load_parquet("baseline_validation_X.parquet")
X_test = load_parquet("baseline_test_X.parquet")
y_train = load_parquet("train_y.parquet")["target_bad"].astype(int)
y_valid = load_parquet("validation_y.parquet")["target_bad"].astype(int)
y_test = load_parquet("test_y.parquet")["target_bad"].astype(int)

for split_name, X_part, y_part in [("train", X_train, y_train), ("validation", X_valid, y_valid), ("test", X_test, y_test)]:
    if len(X_part) != len(y_part):
        raise ValueError(f"{split_name} row mismatch: X={len(X_part)}, y={len(y_part)}")
    if X_part.isna().sum().sum() != 0:
        raise ValueError(f"{split_name} feature matrix contains missing values")

input_summary = pd.DataFrame([
    {"split": "train", "rows": len(X_train), "columns": X_train.shape[1], "bad_rate": round(float(y_train.mean()), 6)},
    {"split": "validation", "rows": len(X_valid), "columns": X_valid.shape[1], "bad_rate": round(float(y_valid.mean()), 6)},
    {"split": "test", "rows": len(X_test), "columns": X_test.shape[1], "bad_rate": round(float(y_test.mean()), 6)},
])
save_table(input_summary, "baseline_modeling_input_summary")
display(input_summary)

Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/tables/baseline_modeling_input_summary.csv


,split,rows,columns,bad_rate
0,train,962641,100,0.188300
1,validation,186920,100,0.246763
2,test,195749,100,0.210315


## 3. Utility Functions

Define class weighting, optional train-only sampling for expensive models, threshold selection on validation, and evaluation metrics.

In [3]:
def compute_sample_weight(y: pd.Series) -> np.ndarray:
    counts = y.value_counts().to_dict()
    n = len(y)
    n_classes = len(counts)
    return y.map({cls: n / (n_classes * count) for cls, count in counts.items()}).astype("float32").to_numpy()


def stratified_train_sample(X: pd.DataFrame, y: pd.Series, max_rows: int, random_state: int) -> tuple[pd.DataFrame, pd.Series]:
    if max_rows is None or len(X) <= max_rows:
        return X, y
    sample_frac = max_rows / len(X)
    sampled_idx = (
        y.to_frame("target_bad")
        .groupby("target_bad", group_keys=False)
        .sample(frac=sample_frac, random_state=random_state)
        .index
    )
    return X.loc[sampled_idx], y.loc[sampled_idx]


def predict_positive_probability(model, X: pd.DataFrame) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    scores = model.decision_function(X)
    return 1 / (1 + np.exp(-scores))


def choose_threshold_by_validation_f1(y_true: pd.Series, y_score: np.ndarray) -> tuple[float, float]:
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    if len(thresholds) == 0:
        return 0.5, 0.0
    f1 = (2 * precision[:-1] * recall[:-1]) / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    best_idx = int(np.nanargmax(f1))
    return float(thresholds[best_idx]), float(f1[best_idx])


def score_split(model_name: str, split: str, y_true: pd.Series, y_score: np.ndarray, threshold: float) -> dict:
    y_pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "model": model_name,
        "split": split,
        "rows": len(y_true),
        "bad_rate": round(float(y_true.mean()), 6),
        "threshold": round(float(threshold), 6),
        "roc_auc": round(float(roc_auc_score(y_true, y_score)), 6),
        "pr_auc": round(float(average_precision_score(y_true, y_score)), 6),
        "brier_score": round(float(brier_score_loss(y_true, y_score)), 6),
        "precision": round(float(precision_score(y_true, y_pred, zero_division=0)), 6),
        "recall": round(float(recall_score(y_true, y_pred, zero_division=0)), 6),
        "f1": round(float(f1_score(y_true, y_pred, zero_division=0)), 6),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def build_score_band_table(model_name: str, split: str, y_true: pd.Series, y_score: np.ndarray, n_bins: int = 10) -> pd.DataFrame:
    df = pd.DataFrame({"target_bad": y_true.to_numpy(), "score": y_score})
    df["score_band"] = pd.qcut(df["score"], q=n_bins, duplicates="drop")
    out = (
        df.groupby("score_band", observed=True)
        .agg(rows=("target_bad", "size"), bad_rate=("target_bad", "mean"), min_score=("score", "min"), max_score=("score", "max"))
        .reset_index()
    )
    out.insert(0, "split", split)
    out.insert(0, "model", model_name)
    out["bad_rate"] = out["bad_rate"].round(6)
    out["min_score"] = out["min_score"].round(6)
    out["max_score"] = out["max_score"].round(6)
    out["score_band"] = out["score_band"].astype(str)
    return out


def fit_and_evaluate_model(model_name: str, model, X_fit: pd.DataFrame, y_fit: pd.Series, use_sample_weight: bool = True) -> tuple[object, pd.DataFrame, pd.DataFrame, dict]:
    start = time.time()
    fit_kwargs = {}
    if use_sample_weight:
        fit_kwargs["sample_weight"] = compute_sample_weight(y_fit)
    model.fit(X_fit, y_fit, **fit_kwargs)
    fit_seconds = round(time.time() - start, 3)

    valid_scores = predict_positive_probability(model, X_valid)
    threshold, best_valid_f1 = choose_threshold_by_validation_f1(y_valid, valid_scores)

    metrics = []
    band_tables = []
    for split, X_part, y_part in [("train", X_train, y_train), ("validation", X_valid, y_valid), ("test", X_test, y_test)]:
        scores = predict_positive_probability(model, X_part)
        metrics.append(score_split(model_name, split, y_part, scores, threshold))
        band_tables.append(build_score_band_table(model_name, split, y_part, scores))

    fit_info = {
        "model": model_name,
        "fit_rows": len(X_fit),
        "fit_columns": X_fit.shape[1],
        "fit_bad_rate": round(float(y_fit.mean()), 6),
        "fit_seconds": fit_seconds,
        "validation_selected_threshold": round(threshold, 6),
        "validation_best_f1": round(best_valid_f1, 6),
    }
    return model, pd.DataFrame(metrics), pd.concat(band_tables, ignore_index=True), fit_info

## 4. Train Logistic Regression

Logistic Regression is the primary interpretable benchmark. It uses class-balanced sample weights and the full chronological training split.

In [4]:
logistic_model = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="saga",
    max_iter=LOGISTIC_MAX_ITER,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=0,
)

logistic_model, logistic_metrics, logistic_bands, logistic_fit_info = fit_and_evaluate_model(
    "logistic_regression", logistic_model, X_train, y_train, use_sample_weight=True
)
display(pd.DataFrame([logistic_fit_info]))
display(logistic_metrics)

,model,fit_rows,fit_columns,fit_bad_rate,fit_seconds,validation_selected_threshold,validation_best_f1
0,logistic_regression,962641,100,0.1883,27.107,0.471834,0.470642


,model,split,rows,bad_rate,threshold,roc_auc,pr_auc,brier_score,precision,recall,f1,tn,fp,fn,tp
0,logistic_regression,train,962641,0.188300,0.471834,0.719952,0.372913,0.214035,0.294519,0.713541,0.416942,471559,309817,51925,129340
1,logistic_regression,validation,186920,0.246763,0.471834,0.695409,0.412338,0.224055,0.355098,0.697648,0.470642,82354,58441,13946,32179
2,logistic_regression,test,195749,0.210315,0.471834,0.700063,0.361512,0.225022,0.310553,0.709296,0.431974,89752,64828,11968,29201


## 5. Train HistGradientBoostingClassifier

Histogram gradient boosting is a strong sklearn tabular baseline. It uses the full chronological training split and class-balanced sample weights.

In [5]:
hgb_model = HistGradientBoostingClassifier(
    loss="log_loss",
    learning_rate=0.06,
    max_iter=HGB_MAX_ITER,
    max_leaf_nodes=31,
    l2_regularization=0.0,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=RANDOM_STATE,
)

hgb_model, hgb_metrics, hgb_bands, hgb_fit_info = fit_and_evaluate_model(
    "hist_gradient_boosting", hgb_model, X_train, y_train, use_sample_weight=True
)
display(pd.DataFrame([hgb_fit_info]))
display(hgb_metrics)

,model,fit_rows,fit_columns,fit_bad_rate,fit_seconds,validation_selected_threshold,validation_best_f1
0,hist_gradient_boosting,962641,100,0.1883,18.311,0.475502,0.47471


,model,split,rows,bad_rate,threshold,roc_auc,pr_auc,brier_score,precision,recall,f1,tn,fp,fn,tp
0,hist_gradient_boosting,train,962641,0.188300,0.475502,0.734196,0.393947,0.209818,0.302432,0.727305,0.427217,477295,304081,49430,131835
1,hist_gradient_boosting,validation,186920,0.246763,0.475502,0.702247,0.424887,0.220547,0.357525,0.706168,0.474710,82263,58532,13553,32572
2,hist_gradient_boosting,test,195749,0.210315,0.475502,0.710063,0.378549,0.216326,0.318053,0.707668,0.438864,92113,62467,12035,29134


## 6. Train Random Forest

Random Forest is included as a diagnostic nonlinear benchmark. To keep runtime practical, fitting uses a stratified sample from the training period only; validation and test evaluation remain full-size.

In [6]:
X_rf_fit, y_rf_fit = stratified_train_sample(X_train, y_train, RF_TRAIN_SAMPLE_ROWS, RANDOM_STATE)

rf_model = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    min_samples_leaf=50,
    max_features="sqrt",
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

rf_model, rf_metrics, rf_bands, rf_fit_info = fit_and_evaluate_model(
    "random_forest", rf_model, X_rf_fit, y_rf_fit, use_sample_weight=False
)
display(pd.DataFrame([rf_fit_info]))
display(rf_metrics)

,model,fit_rows,fit_columns,fit_bad_rate,fit_seconds,validation_selected_threshold,validation_best_f1
0,random_forest,300000,100,0.1883,8.065,0.464443,0.467843


,model,split,rows,bad_rate,threshold,roc_auc,pr_auc,brier_score,precision,recall,f1,tn,fp,fn,tp
0,random_forest,train,962641,0.188300,0.464443,0.732352,0.391485,0.204758,0.297198,0.734819,0.423223,466397,314979,48068,133197
1,random_forest,validation,186920,0.246763,0.464443,0.693471,0.414000,0.214906,0.351448,0.699512,0.467843,81254,59541,13860,32265
2,random_forest,test,195749,0.210315,0.464443,0.702540,0.369670,0.208234,0.313066,0.701134,0.432856,91244,63336,12304,28865


## 7. Compare Models

Compare models across train, validation, and test. Validation selects the classification threshold; test is used once for final temporal holdout performance.

In [7]:
all_fit_info = pd.DataFrame([logistic_fit_info, hgb_fit_info, rf_fit_info])
all_metrics = pd.concat([logistic_metrics, hgb_metrics, rf_metrics], ignore_index=True)
all_score_bands = pd.concat([logistic_bands, hgb_bands, rf_bands], ignore_index=True)

validation_ranking = (
    all_metrics[all_metrics["split"] == "validation"]
    .sort_values(["pr_auc", "roc_auc"], ascending=False)
    .reset_index(drop=True)
)

save_table(all_fit_info, "baseline_model_fit_summary")
save_table(all_metrics, "baseline_model_metrics")
save_table(validation_ranking, "baseline_model_validation_ranking")
save_table(all_score_bands, "baseline_model_score_bands")

display(all_fit_info)
display(all_metrics)
display(validation_ranking)

Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/tables/baseline_model_fit_summary.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/tables/baseline_model_metrics.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/tables/baseline_model_validation_ranking.csv
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/tables/baseline_model_score_bands.csv


,model,fit_rows,fit_columns,fit_bad_rate,fit_seconds,validation_selected_threshold,validation_best_f1
0,logistic_regression,962641,100,0.1883,27.107,0.471834,0.470642
1,hist_gradient_boosting,962641,100,0.1883,18.311,0.475502,0.474710
2,random_forest,300000,100,0.1883,8.065,0.464443,0.467843


,model,split,rows,bad_rate,threshold,roc_auc,pr_auc,brier_score,precision,recall,f1,tn,fp,fn,tp
0,logistic_regression,train,962641,0.188300,0.471834,0.719952,0.372913,0.214035,0.294519,0.713541,0.416942,471559,309817,51925,129340
1,logistic_regression,validation,186920,0.246763,0.471834,0.695409,0.412338,0.224055,0.355098,0.697648,0.470642,82354,58441,13946,32179
2,logistic_regression,test,195749,0.210315,0.471834,0.700063,0.361512,0.225022,0.310553,0.709296,0.431974,89752,64828,11968,29201
3,hist_gradient_boosting,train,962641,0.188300,0.475502,0.734196,0.393947,0.209818,0.302432,0.727305,0.427217,477295,304081,49430,131835
4,hist_gradient_boosting,validation,186920,0.246763,0.475502,0.702247,0.424887,0.220547,0.357525,0.706168,0.474710,82263,58532,13553,32572
5,hist_gradient_boosting,test,195749,0.210315,0.475502,0.710063,0.378549,0.216326,0.318053,0.707668,0.438864,92113,62467,12035,29134
6,random_forest,train,962641,0.188300,0.464443,0.732352,0.391485,0.204758,0.297198,0.734819,0.423223,466397,314979,48068,133197
7,random_forest,validation,186920,0.246763,0.464443,0.693471,0.414000,0.214906,0.351448,0.699512,0.467843,81254,59541,13860,32265
8,random_forest,test,195749,0.210315,0.464443,0.702540,0.369670,0.208234,0.313066,0.701134,0.432856,91244,63336,12304,28865


,model,split,rows,bad_rate,threshold,roc_auc,pr_auc,brier_score,precision,recall,f1,tn,fp,fn,tp
0,hist_gradient_boosting,validation,186920,0.246763,0.475502,0.702247,0.424887,0.220547,0.357525,0.706168,0.474710,82263,58532,13553,32572
1,random_forest,validation,186920,0.246763,0.464443,0.693471,0.414000,0.214906,0.351448,0.699512,0.467843,81254,59541,13860,32265
2,logistic_regression,validation,186920,0.246763,0.471834,0.695409,0.412338,0.224055,0.355098,0.697648,0.470642,82354,58441,13946,32179


## 8. Plots

Save compact comparison plots for validation/test ROC-AUC and PR-AUC.

In [8]:
plot_df = all_metrics[all_metrics["split"].isin(["validation", "test"])].copy()

for metric in ["roc_auc", "pr_auc", "brier_score"]:
    fig, ax = plt.subplots(figsize=(9, 5))
    pivot = plot_df.pivot(index="model", columns="split", values=metric)
    pivot.plot(kind="bar", ax=ax)
    ax.set_title(f"Baseline Model {metric.upper()} By Split")
    ax.set_ylabel(metric)
    ax.set_xlabel("Model")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(title="Split")
    fig.tight_layout()
    path = PLOT_DIR / f"baseline_model_{metric}_comparison.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", path)

Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/plots/baseline_model_roc_auc_comparison.png


Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/plots/baseline_model_pr_auc_comparison.png
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/plots/baseline_model_brier_score_comparison.png


## 9. Export Models

Save trained sklearn models for reproducibility. Large model artifacts should not be committed if the repository ignores `Modeling/modeling_outputs/models/`.

In [9]:
model_paths = []
for name, model in [
    ("logistic_regression", logistic_model),
    ("hist_gradient_boosting", hgb_model),
    ("random_forest", rf_model),
]:
    path = MODEL_DIR / f"{name}.joblib"
    joblib.dump(model, path)
    model_paths.append({"model": name, "path": str(path)})
    print("Saved:", path)

model_paths_df = pd.DataFrame(model_paths)
save_table(model_paths_df, "baseline_model_artifacts")
display(model_paths_df)

Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/models/logistic_regression.joblib
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/models/hist_gradient_boosting.joblib
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/models/random_forest.joblib
Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/tables/baseline_model_artifacts.csv


,model,path
0,logistic_regression,/Users/lindaperez/Documents/NEU/2026SummerML/M...
1,hist_gradient_boosting,/Users/lindaperez/Documents/NEU/2026SummerML/M...
2,random_forest,/Users/lindaperez/Documents/NEU/2026SummerML/M...


## 10. Final Recommendation

Use validation PR-AUC and calibration/error metrics to select a candidate, then confirm on the test split. Do not choose a model based only on train performance.

In [10]:
best_validation_model = validation_ranking.iloc[0]["model"]
recommendation = pd.DataFrame([
    {
        "recommended_model_by_validation_pr_auc": best_validation_model,
        "selection_basis": "Highest validation PR-AUC, with ROC-AUC/Brier/test stability still requiring review.",
        "next_steps": "Review score-band bad rates, probability calibration, and challenger feature set before final model freeze.",
    }
])
save_table(recommendation, "baseline_model_recommendation")
display(recommendation)

Saved: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/tables/baseline_model_recommendation.csv


,recommended_model_by_validation_pr_auc,selection_basis,next_steps
0,hist_gradient_boosting,"Highest validation PR-AUC, with ROC-AUC/Brier/...","Review score-band bad rates, probability calib..."
